# **Márcio Santos (YOLOv5 Real)**

In [1]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import seaborn as sns
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from matplotlib import pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix


In [2]:
DATASET_PATH = "aumentado" 
IMG_SIZE = 128 
EPOCHS = 10
BATCH_SIZE = 32

In [3]:
def load_data_for_torch(path):
    images, labels = [], []
    
    # 1. Obter nomes das classes
    classes = sorted([d for d in os.listdir(path) if os.path.isdir(os.path.join(path, d))])
    class_map = {cls: i for i, cls in enumerate(classes)}
    
    print(f"[Márcio/YOLO] Carregando imagens de {len(classes)} classes...")
    
    for cls in classes:
        cls_dir = os.path.join(path, cls)
        for img_name in os.listdir(cls_dir):
            try:
                img_path = os.path.join(cls_dir, img_name)
                img = cv2.imread(img_path)
                if img is None: continue
                
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                images.append(img)
                labels.append(class_map[cls])
            except Exception: pass
            
    # PyTorch usa canais primeiro: (N, C, H, W) e float 0-1
    X = np.array(images).astype('float32') / 255.0
    X = np.transpose(X, (0, 3, 1, 2)) # Mudando de HWC para CHW
    y = np.array(labels)
    
    return X, y, len(classes), classes

In [4]:
def build_yolo_model(num_classes):
    print("[Márcio/YOLO] Baixando modelo YOLOv5s-cls oficial...")

    try:
        # Usa 'custom' e remove 'pretrained' (foi o que causou o TypeError)
        model = torch.hub.load('ultralytics/yolov5', 'custom', path='yolov5s-cls.pt', force_reload=True)
    except Exception as e:
        print(f"Erro ao baixar: {e}. Tentando fallback (apenas arquitetura local)...")
        # Fallback: Carrega a arquitetura a partir do YAML
        model = torch.hub.load('ultralytics/yolov5', 'custom', path='yolov5s-cls.yaml')
        
    # --- Adaptação da Camada Final (CRÍTICO) ---
    print("Iniciando adaptação da camada de saída...")
    
    try:
        # 1. Acessa a sequência de módulos real dentro do wrapper ClassificationModel
        model_sequence = model.model
        
        # 2. Acessa o último módulo da sequência, que é o Head de Classificação (nn.Sequential ou nn.Linear)
        final_module = model_sequence.model[-1]
        
        # 3. Tenta encontrar a camada nn.Linear dentro do módulo final
        linear_module = None
        if isinstance(final_module, nn.Linear):
            linear_module = final_module
        else:
            # Se for um nn.Sequential, procura o nn.Linear aninhado (fallback)
            for name, module in final_module.named_modules():
                if isinstance(module, nn.Linear):
                    linear_module = module
                    break
        
        if linear_module:
            in_features = linear_module.in_features
            # 4. Substitui a camada Linear original pela nova camada com 'num_classes'
            new_linear = nn.Linear(in_features, num_classes).to(torch.device("cpu")) # Cria na CPU
            
            # Reatribui a nova camada. Se final_module era o nn.Linear, isso falha.
            # Se final_module era um nn.Sequential (o que é comum), precisamos de um set_attr:
            if isinstance(final_module, nn.Linear):
                 model_sequence.model[-1] = new_linear # Substitui diretamente
            else:
                 final_module.add_module('linear', new_linear) # Adiciona no Sequential
        else:
            raise AttributeError("Não foi possível encontrar uma camada nn.Linear para adaptação.")
    
        print(f"Adaptação concluída. Saída alterada para {num_classes} classes.")

    except Exception as e:
        print(f"ERRO CRÍTICO ao adaptar a camada final: {e}")
        print("A estrutura do modelo mudou drasticamente. Verifique a API.")
        exit()
        
    return model

In [5]:
X, y_raw, num_classes, class_names = load_data_for_torch(DATASET_PATH)

if len(X) == 0:
    print("Erro: Nenhuma imagem válida encontrada. Verifique DATASET_PATH.")
    exit()

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y_raw, test_size=0.2, random_state=42)

# Criar Tensores e DataLoaders
train_data = TensorDataset(torch.Tensor(X_train), torch.LongTensor(y_train))
test_data = TensorDataset(torch.Tensor(X_test), torch.LongTensor(y_test))

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE)

# 2. Configurar Modelo
model = build_yolo_model(num_classes)

# Configurar dispositivo (GPU se disponível)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 3. Loop de Treinamento
print("[Márcio/YOLO] Iniciando treinamento...")
train_losses = []

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        
    acc_train = 100 * correct_train / total_train
    avg_loss = running_loss/len(train_loader)
    train_losses.append(avg_loss)
    print(f"Época {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Acc Treino: {acc_train:.2f}%")

# 4. Avaliação e Predição
model.eval()
all_labels = []
all_predictions = []
correct_test = 0
total_test = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        
        # Mover dados para CPU e converter para NumPy para scikit-learn
        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predicted.cpu().numpy())
        
        total_test += labels.size(0)
        correct_test += (predicted == labels).sum().item()

final_accuracy = correct_test / total_test
print(f"\n>>> Acurácia Final (Abordagem Márcio/YOLOv5): {100 * final_accuracy:.2f}%")

[Márcio/YOLO] Carregando imagens de 25 classes...
[Márcio/YOLO] Baixando modelo YOLOv5s-cls oficial...
Downloading: "https://github.com/ultralytics/yolov5/zipball/master" to /Users/lucatvillela/.cache/torch/hub/master.zip


/Users/lucatvillela/Documents/facul/venvFacul/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
YOLOv5 🚀 2025-12-2 Python-3.9.6 torch-2.8.0 CPU

Fusing layers... 
Model summary: 117 layers, 5447688 parameters, 0 gradients, 11.4 GFLOPs
WARNING ⚠️ YOLOv5 ClassificationModel is not yet AutoShape compatible. You must pass torch tensors in BCHW to this model, i.e. shape(1,3,224,224).


Iniciando adaptação da camada de saída...
Adaptação concluída. Saída alterada para 25 classes.
Usando dispositivo: cpu
[Márcio/YOLO] Iniciando treinamento...
Época 1/10 | Loss: 1.8490 | Acc Treino: 50.93%
Época 2/10 | Loss: 0.7751 | Acc Treino: 83.50%
Época 3/10 | Loss: 0.5121 | Acc Treino: 90.09%
Época 4/10 | Loss: 0.3743 | Acc Treino: 93.45%
Época 5/10 | Loss: 0.2864 | Acc Treino: 95.84%
Época 6/10 | Loss: 0.2332 | Acc Treino: 96.46%
Época 7/10 | Loss: 0.1914 | Acc Treino: 97.70%
Época 8/10 | Loss: 0.1575 | Acc Treino: 98.85%
Época 9/10 | Loss: 0.1351 | Acc Treino: 98.94%
Época 10/10 | Loss: 0.1158 | Acc Treino: 99.25%

>>> Acurácia Final (Abordagem Márcio/YOLOv5): 91.50%


In [6]:
print("\n--- Relatório de Classificação por Classe (YOLOv5) ---")
print(classification_report(all_labels, all_predictions, target_names=class_names))


--- Relatório de Classificação por Classe (YOLOv5) ---
              precision    recall  f1-score   support

           a       1.00      0.95      0.98        21
           b       0.95      0.82      0.88        22
           c       1.00      1.00      1.00        21
           d       0.89      0.86      0.88        29
           e       0.96      0.87      0.91        30
           f       0.71      0.91      0.80        22
           g       0.89      0.85      0.87        20
       h e k       0.80      1.00      0.89        20
           i       0.82      0.93      0.88        15
           j       1.00      0.96      0.98        27
           l       0.96      1.00      0.98        22
           m       0.92      0.85      0.88        26
           n       0.76      1.00      0.86        25
           o       1.00      0.85      0.92        20
           p       1.00      0.95      0.98        21
           q       0.95      0.84      0.89        25
           r       0.90  

In [9]:
print("Gerando imagem da Matriz de Confusão...")
try:
    cm = confusion_matrix(all_labels, all_predictions)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Matriz de Confusão - YOLOv5 (PyTorch)')
    plt.ylabel('Verdadeiro')
    plt.xlabel('Predito')
    plt.tight_layout()
    
    # Salvar arquivo
    plt.savefig('matriz_confusao_yolo.png')
    print("Arquivo salvo: matriz_confusao_yolo.png")
    plt.close()
except Exception as e:
    print(f"Erro ao salvar Matriz: {e}")

Gerando imagem da Matriz de Confusão...
Arquivo salvo: matriz_confusao_yolo.png


In [10]:
print("Gerando imagem do Gráfico de Perda...")
try:
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Treino', color='red', linewidth=2)
    plt.title('Evolução da Perda (Loss) - YOLOv5')
    plt.xlabel('Época')
    plt.ylabel('CrossEntropy Loss')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    
    # Salvar arquivo
    plt.savefig('grafico_perda_yolo.png')
    print("Arquivo salvo: grafico_perda_yolo.png")
    plt.close()
except Exception as e:
    print(f"Erro ao salvar Loss: {e}")

Gerando imagem do Gráfico de Perda...
Arquivo salvo: grafico_perda_yolo.png


In [11]:
torch.save(model.state_dict(), 'modelo_marcio_weights.pth')